**Task 2 Data Science Job Simulation**

In [2]:
import pandas as pd

In [10]:
from google.colab import files
uploaded = files.upload()

Saving mobile_customers.xlsx to mobile_customers.xlsx


In [12]:
import io
data = io.BytesIO(uploaded['mobile_customers.xlsx'])

In [15]:
import uuid
df = pd.read_excel('mobile_customers.xlsx')

print("Dimensões originais:", df.shape)
print("Colunas:", df.columns.tolist())


Dimensões originais: (10000, 19)
Colunas: ['Unnamed: 0', 'customer_id', 'date_registered', 'username', 'name', 'gender', 'address', 'email', 'birthdate', 'current_location', 'residence', 'employer', 'job', 'age', 'salary', 'credit_card_provider', 'credit_card_number', 'credit_card_security_code', 'credit_card_expire']


In [16]:
# Cria uma chave aleatória por registro
# Não revela o custumer_id
anon = pd.DataFrame()
anon["customer_key"] = [
    str(uuid.uuid4()) for _ in range(len(df))
]

In [17]:
# Generaliza data de cadastro em períodos
datas = pd.to_datetime(df["date_registered"], errors="coerce")
ano = datas.dt.year

anon["registration_period"] = pd.cut(
    ano,
    bins=[2018, 2020, 2022, 2024],
    labels=["2018-2020", "2020-2022", "2022-2024"],

)

In [18]:
anon["gender"] = df["gender"]

In [20]:
# Generaliza idade em faixas
anon["age_band"] = pd.cut(
    df["age"],
    bins=[0, 29, 39, 49, 59, 69, float("inf")],
    labels=[
        "18-29",
        "30-39",
        "40-49",
        "50-59",
        "60-69",
        "70+"
    ],
    include_lowest=True,
)

In [21]:
# Generaliza Salario
salario = pd.to_numeric(
    df["salary"], errors="coerce"
)
anon["salary_band"] = pd.cut(
    salario,
    bins=[
        -float("inf"),
        49999,
        99999,
        149999,
        float("inf"),
    ],
    labels=[
        "Under 50k",
        "50-100k",
        "100-150k",
        "Over 150k",
    ]
)

In [22]:
# Exportar para CSV
arquivo_saida = "mobile_customers.xlsx"

anon.to_csv(arquivo_saida, index=False, encoding= "utf-8")
print(f"Arquivo de saída gerado: {arquivo_saida}")

Arquivo de saída gerado: mobile_customers.xlsx


In [23]:
# Validar anonimização

colunas_proibidas = [
    "customer_id",
    "name",
    "username",
    "email",
    "address",
    "birthdate",
    "current_location",
    "residence",
    "employer",
    "job",
    "salary",
    "credit_card_provider",
    "credit_card_number",
    "credit_card_security_code",
    "credit_card_expire"
]

presentes = [
    coluna for coluna in colunas_proibidas
    if coluna in anon.columns
]

assert not presentes, (
    f"Campos sensíveis encontrados: {presentes}"
)

In [24]:
assert len(anon) == len(df), (
    "Número de registros não corresponde"
)
print("Quantidade de registros:", len(anon))
print("Quantidade de colunas:", len(anon.columns))
print("Chaves únicas:", anon["customer_key"].nunique())

print("\nValores ausentes por coluna:")
print(anon.isna().sum())


Quantidade de registros: 10000
Quantidade de colunas: 5
Chaves únicas: 10000

Valores ausentes por coluna:
customer_key           0
registration_period    0
gender                 0
age_band               0
salary_band            0
dtype: int64


In [25]:
# Analisar grupos de reidentificação


quasi_identificadores = [
    "registration_period",
    "gender",
    "age_band",
    "salary_band"
]

tamanhos_grupos = anon.groupby(
    quasi_identificadores,
    observed=True,
    dropna=False
).size()

if len(tamanhos_grupos) > 0:
    print(
        "\nMenor grupo de atributos semelhantes:",
        tamanhos_grupos.min()
    )
else:
    print("\nNão foi possível calcular os grupos.")

print("\nPrévia do resultado:")
print(anon.head())

print("\nProcessamento concluído.")



Menor grupo de atributos semelhantes: 38

Prévia do resultado:
                           customer_key registration_period gender age_band  \
0  ce3a195d-7c1e-4922-8613-8c7a186a3a4a           2020-2022      M    40-49   
1  153b03f3-3f9d-43de-92f3-643daea9a463           2018-2020      F    40-49   
2  12f4058c-f009-4c42-a8dd-6e947ee5ccfc           2018-2020      M    40-49   
3  510e7073-ce44-4e4c-9245-8c4376b7cb17           2020-2022      F    30-39   
4  42f7e495-0efe-494d-a7e6-dbf7577762d3           2018-2020      F    50-59   

  salary_band  
0     50-100k  
1     50-100k  
2   Over 150k  
3    100-150k  
4    100-150k  

Processamento concluído.


In [26]:
files.download("mobile_customers.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>